# 06. LLM Agent — 自动化分析报告

职责4-AI 探索：探索大模型在数据分析场景的应用，参与 AI Agent 产品设计。

涵盖：可插拔 LLM 接口、自动报告生成、端到端 Agent 编排。

## 1. 可插拔 LLM 接口

设计 `BaseLLMProvider` 抽象接口，支持 Mock/Demo 模式，预留 OpenAI 等接入点。

In [ ]:
import sys; sys.path.insert(0, '../src')
from llm_agent import get_provider, MockLLMProvider, ReportGenerator

# Mock LLM（无需 API Key）
llm = get_provider('mock')
print('Provider:', llm.name, '| available:', llm.is_available())

resp = llm.chat('分析销量异常的原因')
print(resp.content[:300])

## 2. 自动报告生成

In [ ]:
from utils.data_loader import prepare_all
from attribution import dimension_contribution
from anomaly import StatisticalAnomalyDetector
import pandas as pd

ds = prepare_all()
byd = ds['byd_monthly']
reporter = ReportGenerator(MockLLMProvider())

# 异常检测报告
series = byd.set_index('month')['sales']
det = StatisticalAnomalyDetector('sigma', k=2).detect(series)
report = reporter.generate_anomaly_report(series, det)
print(report)

## 3. 端到端 Agent 编排

数据 → 探索 → 建模 → 异常 → 归因 → 报告 全流程自动化。

In [ ]:
from llm_agent import AnalysisAgent, MockLLMProvider

md = ds['multi_dim_sales'].copy()
md['period'] = md['month'].dt.strftime('%Y-%m')
agent = AnalysisAgent(
    df=md, metric_col='sales', period_col='period', time_col='month',
    llm=MockLLMProvider()
)
report = agent.run_all()
display(agent.summary())
print('\n=== 综合报告 ===')
print(report[:500])

## 4. 接入真实 LLM

设置环境变量后即可切换到真实模型：
```python
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
from llm_agent import get_provider
llm = get_provider('openai', model='gpt-4o-mini')
```

## 小结

- 可插拔设计：Mock 模式离线可用，生产环境接入真实 LLM
- Agent 编排：端到端自动化，减少人工分析环节
- 报告生成：将结构化分析结果转化为自然语言业务报告